In [1]:
!find /kaggle/input -name "common.py"

/kaggle/input/datasets/nguynchongqun/ewc-code/ewc-project-main/common.py


In [2]:
import torch
print(torch.cuda.is_available())  # Kết quả phải là True
print(torch.cuda.get_device_name(0)) # In ra tên GPU

True
Tesla T4


In [3]:
import glob, os, shutil, sys

WORK = "/kaggle/working"
REPO = f"{WORK}/ewc-project-main"

if not os.path.exists(f"{REPO}/common.py"):
    hits = glob.glob("/kaggle/input/**/common.py", recursive=True)
    assert hits, "Không thấy common.py trong /kaggle/input, kiểm tra lại dataset đã Add Input chưa."
    src = os.path.dirname(hits[0])
    print("Tìm thấy code ở:", src)
    shutil.copytree(src, REPO)

os.chdir(REPO)
sys.path.insert(0, REPO)          # insert(0), không dùng append

import common
print(common.__file__, common.DEVICE, common.ROOT)
print(sorted(os.listdir(REPO)))

Tìm thấy code ở: /kaggle/input/datasets/nguynchongqun/ewc-code/ewc-project-main
/kaggle/working/ewc-project-main/common.py cuda /kaggle/working
['.gitignore', 'README.md', '__pycache__', 'common.py', 'data.py', 'eval.py', 'ewc.py', 'fakes.py', 'fisher.py', 'metrics.py', 'test_common.py', 'test_data.py', 'test_eval.py', 'test_ewc.py', 'test_fisher.py', 'test_interface.py']


In [4]:
import json, time
from common import make_model, RESULTS_DIR
from data import get_tasks
from ewc import train_sequential
from eval import save_result, load_results
from metrics import avg_acc

# ====== CHỈ ĐỔI DÒNG NÀY: "sgd" (Quân) | "l2" (Hưng) | "ewc" (Kiệt) ======
METHOD = "sgd"
# =========================================================================

KIND, N_TASKS = "permuted", 10
LR, EPOCHS, BATCH = 0.1, 10, 128          # cấu hình chung đã chốt ở issue #6
VAL_SEED = 0                               # sweep λ chỉ dùng 1 seed
FINAL_SEEDS = [0, 1, 2]

# Lưới λ. L2: lr*λ*9 < 2 -> λ < ~2.2. EWC: Fisher nhỏ nên λ lớn hơn nhiều.
# Run nào phân kỳ sẽ được ghi nhận và bỏ qua, không làm dừng notebook.
LAM_GRID = {
    "sgd": [0.0],
    "l2":  [0.01, 0.03, 0.1, 0.3, 1.0, 2.0],
    "ewc": [1.0, 10.0, 100.0, 1000.0, 10000.0],
}[METHOD]

VAL_DIR = os.path.join(RESULTS_DIR, "val")
DIV_DIR = os.path.join(RESULTS_DIR, "diverged")
os.makedirs(DIV_DIR, exist_ok=True)


def run_one(holdout: str, seed: int, lam: float):
    """Chạy 1 run. Trả về avg acc, hoặc None nếu phân kỳ. Bỏ qua nếu file kết quả đã có (chạy lại được)."""
    out_dir = VAL_DIR if holdout == "val" else RESULTS_DIR
    path = os.path.join(out_dir, f"{KIND}_{METHOD}_lam{lam:g}_seed{seed}.json")
    if os.path.exists(path):
        with open(path) as f:
            a = avg_acc(json.load(f)["acc"])
        print(f"[skip] {os.path.basename(path)} (avg acc {a:.4f})")
        return a

    tasks, task_classes = get_tasks(KIND, N_TASKS, batch_size=BATCH, seed=seed,
                                    holdout=holdout, val_frac=0.1)
    model = make_model(seed)
    t0 = time.time()
    try:
        acc = train_sequential(model, tasks, task_classes, METHOD, lam, EPOCHS, lr=LR)
    except FloatingPointError as e:
        print(f"[diverged] {METHOD} lam={lam:g} seed={seed} holdout={holdout}: {e}")
        with open(os.path.join(DIV_DIR, f"{KIND}_{METHOD}_lam{lam:g}_seed{seed}_{holdout}.json"), "w") as f:
            json.dump({"method": METHOD, "lam": lam, "seed": seed, "holdout": holdout,
                       "lr": LR, "epochs": EPOCHS, "error": str(e)}, f)
        return None

    runtime = time.time() - t0
    params = {"lr": LR, "epochs": EPOCHS, "batch_size": BATCH, "holdout": holdout, "runtime_s": runtime}
    save_result(KIND, METHOD, lam, seed, acc, params, out_dir=out_dir)
    a = avg_acc(acc)
    print(f"[done] {METHOD} lam={lam:g} seed={seed} holdout={holdout} "
          f"avg acc={a:.4f} task0={acc[-1][0]:.4f} ({runtime/60:.1f} phút)")
    return a

In [5]:
import time
import ewc as _ewc_mod

_orig_evaluate = _ewc_mod.evaluate
_prog = {"n": 0, "last": time.time()}

def _logged_evaluate(model, loaders, classes):
    accs = _orig_evaluate(model, loaders, classes)
    _prog["n"] += 1
    k = (_prog["n"] - 1) % N_TASKS            # task vừa học xong (0..9)
    now = time.time()
    print(f"   task {k+1}/{N_TASKS} xong | {(now-_prog['last'])/60:.1f} phút | "
          f"acc task đó={accs[k]:.4f} | acc TB các task đã học={sum(accs[:k+1])/(k+1):.4f}",
          flush=True)
    _prog["last"] = now
    return accs

_ewc_mod.evaluate = _logged_evaluate

In [6]:
val_scores = {}
if METHOD != "sgd":
    for lam in LAM_GRID:
        val_scores[lam] = run_one("val", VAL_SEED, lam)

    print("\nAvg acc validation theo λ:")
    for lam, a in val_scores.items():
        print(f"  λ={lam:g}: {'diverged' if a is None else f'{a:.4f}'}")
else:
    print("SGD không có λ, bỏ qua sweep.")

SGD không có λ, bỏ qua sweep.


In [7]:
if METHOD == "sgd":
    BEST_LAM = 0.0
else:
    val_res = [r for r in load_results(VAL_DIR)
               if r["kind"] == KIND and r["method"] == METHOD
               and r.get("holdout") == "val" and r["seed"] == VAL_SEED]
    assert val_res, "Chưa có kết quả validation, hãy chạy Cell 3 trước."
    scored = sorted(((avg_acc(r["acc"]), r["lam"]) for r in val_res), reverse=True)
    BEST_LAM = scored[0][1]
    print("λ (val avg acc):", [(f"{l:g}", round(a, 4)) for a, l in scored])
    if BEST_LAM in (min(LAM_GRID), max(LAM_GRID)) and len(LAM_GRID) > 1:
        print("⚠ λ tốt nhất nằm ở biên lưới, nên mở rộng LAM_GRID rồi chạy lại Cell 3.")

print(f"\n==> {METHOD}: BEST_LAM = {BEST_LAM:g}")
with open(os.path.join(RESULTS_DIR, f"best_lam_{METHOD}.json"), "w") as f:
    json.dump({"method": METHOD, "lam": BEST_LAM, "selected_on": "val", "val_seed": VAL_SEED}, f)


==> sgd: BEST_LAM = 0


In [8]:
final = {}
for seed in FINAL_SEEDS:
    final[seed] = run_one("test", seed, BEST_LAM)

ok = [a for a in final.values() if a is not None]
print(f"\n{METHOD} λ={BEST_LAM:g}: final test avg acc = "
      f"{sum(ok)/len(ok):.4f} trên {len(ok)}/{len(FINAL_SEEDS)} seed")

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.82MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 160kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.30MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.77MB/s]


   task 1/10 xong | 1.8 phút | acc task đó=0.9742 | acc TB các task đã học=0.9742
   task 2/10 xong | 1.5 phút | acc task đó=0.9804 | acc TB các task đã học=0.9702
   task 3/10 xong | 1.5 phút | acc task đó=0.9800 | acc TB các task đã học=0.9596
   task 4/10 xong | 1.5 phút | acc task đó=0.9782 | acc TB các task đã học=0.9376
   task 5/10 xong | 1.5 phút | acc task đó=0.9801 | acc TB các task đã học=0.9129
   task 6/10 xong | 1.5 phút | acc task đó=0.9824 | acc TB các task đã học=0.8763
   task 7/10 xong | 1.5 phút | acc task đó=0.9805 | acc TB các task đã học=0.8659
   task 8/10 xong | 1.5 phút | acc task đó=0.9798 | acc TB các task đã học=0.8105
   task 9/10 xong | 1.5 phút | acc task đó=0.9768 | acc TB các task đã học=0.7431
   task 10/10 xong | 1.5 phút | acc task đó=0.9792 | acc TB các task đã học=0.7326
[done] sgd lam=0 seed=0 holdout=test avg acc=0.7326 task0=0.4865 (15.4 phút)
   task 1/10 xong | 1.5 phút | acc task đó=0.9748 | acc TB các task đã học=0.9748
   task 2/10 xong | 

In [9]:
archive = shutil.make_archive(f"{WORK}/results_{METHOD}", "zip", RESULTS_DIR)
print("Tải file này từ tab Output:", archive)

Tải file này từ tab Output: /kaggle/working/results_sgd.zip
